# PMAPS Workshop: IDAES-GTEP, Session 2

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/agmoore4/idaes-gtep.git/pmaps-123?urlpath=%2Fdoc%2Ftree%2Fdocs%2Fsource%2Ftutorials%2F123bus%2Ftutorial_123bus.ipynb)

Welcome! In this tutorial, we will demonstrate using IDAES Generation and Transmission Expansion Planning (GTEP) with a more complex case--a 123-bus system in Texas.

As we step through the notebook, you should notice that all the steps to set up and solve a model are the same as in the simpler 5-bus case. The only major difference is the scale of the system reflected in the data files. However, we will demonstrate some helpful (optional) functionality for larger cases that take longer to solve.

In [33]:
# suppressing some logs/warnings
import logging
import warnings
logging.getLogger().setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

## Setting up and solving the model

First, we create our `ExpansionPlanningData` instance and use `load_prescient` to load the relevant data files.

In [34]:
from pathlib import Path
from gtep.gtep_data import ExpansionPlanningData

data_path = Path("../../../../gtep/data/123_Bus_Resil_Week")

rep_days = [
    "2019-01-28 00:00",
    # "2019-07-05 00:00",
]
rep_weights = {
    "2019-01-28 00:00": 365,
    # "2019-01-28 00:00": 210,
    # "2019-07-05 00:00": 155,
}

data_object = ExpansionPlanningData(
    stages=1,
    num_reps=1,
    num_commit=1,
    num_dispatch=1,
    duration_representative_period=0.25,  # keeping dispatch periods 15 min
)
data_object.load_prescient(
    data_path,
    representative_dates=rep_days,
    representative_weights=rep_weights,
)

Setting default t0 state in RTS-GMLC parser


Next we read in cost data using `DataProcessing`:

In [35]:
from gtep.gtep_data_processing import DataProcessing

bus_data_path = Path(
    "../../../../gtep/data/costs/Bus_data_gen_weights_mappings.csv"
)
cost_data_path = Path(
    "../../../../gtep/data/costs/2022_v3_Annual_Technology_Baseline_Workbook_Mid-year_update_2-15-2023_Clean.xlsx"
)
ng_cost_path = Path(
    "../../../../gtep/data/costs/Total_Energy_Supply_Disposition_and_Price_Summary.csv"
)

candidate_gens = [
    "Natural Gas_FE",
    "Solar - Utility PV",
    "Land-Based Wind",
]

cost_data = DataProcessing()
cost_data.load_gen_data(
    bus_data_path=bus_data_path,
    cost_data_path=cost_data_path,
    ng_cost_path=ng_cost_path,
    candidate_gens=candidate_gens,
)

Finally, we create and solve the model. However, we are doing two small (optional) things differently compared to Session 1.

1. Since the 123-bus case is much larger than the 5-bus case, it can be useful to track how much time elapses between each step (build, transform, and solve). To do so, we can make use of the `ExpansionPlanningModel`'s `.timer` attribute, which is a `TicTocTimer` instance (see the Pyomo docs for it here: https://pyomo.readthedocs.io/en/stable/api/pyomo.common.timing.TicTocTimer.html).
<br> In short, calling `mod_object.timer.toc()` will report the elapsed time since the last `.tic()`/`.toc()`. Note that an `ExpansionPlanningModel` automatically calls `self.timer.tic()` when `.create_model()` is called on it (with the message `"Creating GTEP Model"`).

2. For models that take longer to solve, it can be useful to pass specific arguments to the solver. For instance, below we choose to pass values for `user_objective_scale` and `user_bound_scale`, which can improve the solve time of the optimizer.

In [36]:
from gtep.gtep_model import ExpansionPlanningModel
from pyomo.environ import SolverFactory, TransformationFactory
# from contextlib import redirect_stdout, redirect_stderr

mod_object = ExpansionPlanningModel(
    data=data_object,
    cost_data=cost_data,
    config={"scale_loads": False},
)
mod_object.create_model()
mod_object.timer.toc("Finished model build")

TransformationFactory("gdp.bigm").apply_to(mod_object.model)
mod_object.timer.toc("Finished model transformation")

opt = SolverFactory("highs")
opt.options["user_objective_scale"] = -7
opt.options["user_bound_scale"] = -5

# FOR TESTING THE NOTEBOOK, HAVING PRINT TO CELL OUTPUT. BEFORE WORKSHOP, UNCOMMENT THE `with` STATEMENT (OR REMOVE `tee=True`)
# with open("output.log", "w") as f, redirect_stdout(f), redirect_stderr(f):
result = opt.solve(mod_object.model, tee=True)

mod_object.timer.toc("Finished solving");

[    0.00] Creating GTEP Model
[+   0.48] Finished model build
[+   0.41] Finished model transformation
Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
MIP has 8044 rows; 7092 cols; 21595 nonzeros; 3450 integer variables (2971 binary)
Assessing costs and bounds after applying user_objective_scale option value of -7 and user_bound_scale option value of -5
Coefficient ranges:
  Matrix  [3e-02, 3e+06]
  Cost    [8e-03, 9e+05]
  Bound   [3e-02, 1e+02]
  RHS     [1e-02, 3e+05]
Presolving model
3871 rows, 3050 cols, 10774 nonzeros 0s
3315 rows, 2413 cols, 9048 nonzeros 0s
3314 rows, 2412 cols, 9058 nonzeros 0s
Presolve reductions: rows 3314(-4730); columns 2412(-4680); nonzeros 9058(-12537) 

Solving MIP model with:
   3314 rows
   2412 cols (1480 binary, 0 integer, 0 implied int., 932 continuous, 0 domain fixed)
   9058 nonzeros
   Thread count 4 (of 8 threads). Using 1

Now we use the `ExpansionPlanningSolution` class to write out the results and generate plots:

In [ ]:
from copy import copy
from IPython.display import display, HTML
from gtep.gtep_solution import ExpansionPlanningSolution

def display_plotly_as_html(fig, title=None):
    if title is not None:
        fig = copy(fig)
        fig.update_layout(title=title)
    fig_html = fig.to_html(full_html=False, include_plotlyjs="cdn")
    return display(HTML(fig_html))

# create solution object and write out to json
soln = ExpansionPlanningSolution(data_path)
soln_path = Path("soln")
soln.save_results_in_json_files(mod_object, soln_path)

# perform plotting
pie = soln.create_plots("combined", soln_path, data_path, "piechart", savefig=False)
stackgraph = soln.create_stackgraph(soln_path, rep_days, savefig=False)

display_plotly_as_html(pie, title="Baseline 123-bus case")
# display_plotly_as_html(stackgraph, title="Baseline 123-bus case")  # uncomment once stackgraph function is fixed

[Markdown discussing outputs here]

## Experiments

Here we can again define a helper function to solve the model and return plots:

In [48]:
from pyomo.environ import value

def solve_model_and_make_plots(
    model_object: ExpansionPlanningModel,
    data_path: Path|str,
    write_dir: Path|str,
    tee: bool=True,
    solver_options: None|dict=None,
):
    
    # transform and solve
    TransformationFactory("gdp.bigm").apply_to(model_object.model)
    model_object.timer.toc("Finished model transformation")

    opt = SolverFactory("highs")
    if solver_options is not None:
        for option, val in solver_options.items():
            opt.options[option] = val

    result_object = opt.solve(model_object.model, tee=tee)
    model_object.timer.toc("Finished solving")

    # check termination condition
    term_cond = result_object["Solver"][0]["Termination condition"]
    print("Termination condition:", term_cond)
    if term_cond != "optimal":
        return

    # make solution
    soln_object = ExpansionPlanningSolution(data_path)
    soln_object_path = (Path() / write_dir).resolve()
    soln_object.save_results_in_json_files(model_object, soln_object_path)
    model_object.timer.toc("Finished saving results to json")
    pie = soln_object.create_plots(
        "combined", soln_object_path, data_path, "piechart", savefig=False
    )
    rep_days = [
        value(model_object.model.representativeDate[idx])
        for idx in model_object.model.representativeDate
    ]
    stackgraph = soln_object.create_stackgraph(
        soln_object_path, rep_days, savefig=False
    )
    model_object.timer.toc("Finished plotting")
    return pie, stackgraph

### Experiment 1

In this experiment, we increase the number of representative periods to 2, which allows us to consider how different renewable availability and load profiles result in different model choices.

In [ ]:
data_object_exper1 = ExpansionPlanningData(
    stages=1,
    num_reps=2,
    num_commit=1,
    num_dispatch=1,
    duration_representative_period=0.25,
)

data_object_exper1.load_prescient(
    data_path,
    representative_dates=[
        "2019-01-28 00:00",
        "2019-07-05 00:00",
    ],
    representative_weights={
        "2019-01-28 00:00": 210,
        "2019-07-05 00:00": 155,
    },
)

mod_object_expr1 = ExpansionPlanningModel(
    data=data_object_exper1,
    cost_data=cost_data,  # same cost data
    config={"scale_loads": False},
)
mod_object_expr1.create_model()
mod_object_expr1.timer.toc("Finished model build")

pie_exper1, stackgraph_exper1 = solve_model_and_make_plots(
    mod_object_expr1,
    data_path,
    "soln_two_repr",
)

print(
    "Objective for baseline:",
    value(mod_object.model.total_cost_objective)
)
print(
    "Objective for experiment 1:",
    value(mod_object_expr1.model.total_cost_objective)
)

display_plotly_as_html(pie, title="Baseline")
display_plotly_as_html(pie_exper1, title="Experiment 1 (2 representative periods)")

# display_plotly_as_html(stackgraph, title="Baseline")
# display_plotly_as_html(stackgraph_exper1, title="Experiment 1 (2 representative periods)")

Setting default t0 state in RTS-GMLC parser
[    0.00] Creating GTEP Model
[+   0.60] Finished model build
[+   0.68] Finished model transformation
Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
MIP has 15504 rows; 11612 cols; 41605 nonzeros; 5675 integer variables (5196 binary)
Coefficient ranges:
  Matrix  [1e+00, 3e+06]
  Cost    [1e+00, 4e+09]
  Bound   [1e+00, 4e+03]
  RHS     [3e-01, 1e+07]
Presolving model
8127 rows, 6053 cols, 22320 nonzeros 0s
6864 rows, 4943 cols, 18568 nonzeros 0s
6861 rows, 4940 cols, 18596 nonzeros 0s
Presolve reductions: rows 6861(-8643); columns 4940(-6672); nonzeros 18596(-23009) 

Solving MIP model with:
   6861 rows
   4940 cols (3077 binary, 0 integer, 0 implied int., 1863 continuous, 0 domain fixed)
   18596 nonzeros
   Thread count 4 (of 8 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central ro

In this experiment, adding a second representative period resulted in slightly more investment into Wind and Solar capacity (as seen in the pie chart).

[Flesh out markdown, including discussing stackgraphs]

### Experiment 2

In this experiment, we showcase solving the model for many more time periods [describe specifics here].

The model takes several hours to run, so we will show how to set it up but won't solve it here. Instead, we will load solution jsons from a previous run to produce figures.

In [ ]:
data_object_exper2 = ExpansionPlanningData(
    stages=1,
    num_reps=6,
    num_commit=6,
    num_dispatch=4,
    duration_representative_period=1,
)

# THIS IS AN EXAMPLE; REPLACE WITH THE ACTUAL RUN PARAMETERS
data_object_exper2.load_prescient(
    data_path,
    representative_dates=[
        "2019-01-28 00:00",
        "2019-03-28 00:00",
        "2019-05-28 00:00",
        "2019-07-05 00:00",
        "2019-09-05 00:00",
        "2019-11-05 00:00",
    ],
    representative_weights={
        "2019-01-28 00:00": 60,
        "2019-03-28 00:00": 61,
        "2019-05-28 00:00": 61,
        "2019-07-05 00:00": 61,
        "2019-09-05 00:00": 61,
        "2019-11-05 00:00": 61,
    },
)

mod_object_expr2 = ExpansionPlanningModel(
    data=data_object_exper2,
    cost_data=cost_data,  # same cost data
    config={"scale_loads": False},
)
mod_object_expr1.create_model()
mod_object_expr1.timer.toc("Finished model build")

print("-" * 50)
print("We then could continue on to solve...")

Setting default t0 state in RTS-GMLC parser
[    0.00] Creating GTEP Model
[+   2.41] Finished model build
--------------------------------------------------
We then could continue on to solve...


In [ ]:
saved_result_dir = ""

soln_object_exper2 = ExpansionPlanningSolution(data_path)
soln_object_path_exper2 = (Path() / saved_result_dir).resolve()

pie_exper2 = soln_object_exper2.create_plots(
    "combined", soln_object_path_exper2, data_path, "piechart", savefig=False
)
rep_days = [
    value(mod_object_expr2.model.representativeDate[idx])
    for idx in mod_object_expr2.model.representativeDate
]
stackgraph = soln_object_exper2.create_stackgraph(
    soln_object_path_exper2, rep_days, savefig=False
)

display_plotly_as_html(pie, title="Baseline")
display_plotly_as_html(pie_exper2, title="Experiment 1 (2 representative periods)")

# display_plotly_as_html(stackgraph, title="Baseline")
# display_plotly_as_html(stackgraph_exper1, title="Experiment 1 (2 representative periods)")

[Discuss results here]